In [ ]:
from database.manager import DatabaseManager
from analysis.engine import Engine
from analysis.seasonality import Seasonality
from analysis.plotting import SeasonalityPlotter

# 1. Setup
db = DatabaseManager()
engine = Engine()
seaso = Seasonality()
plotter = SeasonalityPlotter()

# 2. Params
# symbol = "CL"
# expression = "Z26 - 2*F27 + G27"
# seasons = [2023, 2024, 2025, 2026]

# # 3. Pipeline
# codes = engine.get_required_codes(expression, seasons)
# raw = db.get_contract_history(symbol, codes)
# prep = seaso.prepare_calendar(raw)
# calc = seaso.calculate_expression(prep, expression, symbol)
# aligned = seaso.align_to_expiry(calc, window_days=250)

# # 4. Add Average & Plot
# aligned["AVERAGE"] = seaso.get_average(aligned)
# plotter.plot(aligned, title=f"{symbol} Butterfly Seasonality")

In [2]:
# ================================================
# DEBUGGING BLOCK
# ================================================
import sqlite3
from database.manager import DB_PATH

# 1. Check if Database has anything
conn = sqlite3.connect(DB_PATH)
count = conn.execute("SELECT count(*) FROM seac_settlements").fetchone()[0]
print(f"Total rows in Database: {count}")

# 2. Check the codes the Engine generated
codes = engine.get_required_codes(expression, seasons)
print(f"Codes needed: {codes}")

# 3. Check if raw data was actually fetched
raw = db.get_contract_history(symbol, codes)
print(f"Contracts found in DB: {list(raw.keys())}")

# 4. Check if calculation worked
prep = seaso.prepare_calendar(raw)
calc = seaso.calculate_expression(prep, expression, symbol)
print(f"Seasons successfully calculated: {list(calc.keys())}")

# 5. Check if alignment worked
aligned = seaso.align_to_expiry(calc, window_days=250)
print(f"Seasons after alignment: {list(aligned.keys())}")

# 6. Finally try average if aligned is not empty
if not aligned:
    print("ERROR: No data left to plot. Check if the contracts overlap on the same dates.")
else:
    avg = seaso.get_average(aligned)
    if not avg.empty:
        aligned["AVERAGE"] = avg
        plotter.plot(aligned, title=f"{symbol} Butterfly Seasonality")

Total rows in Database: 1076557


NameError: name 'expression' is not defined

In [4]:
# ================================================
# FINAL PLOT: CL Butterfly Z-2F+G
# ================================================

# 1. Add the Average line to your aligned data
# This calculates the historical "norm" for this butterfly
aligned["AVERAGE"] = seaso.get_average(aligned)

# 2. Plot using the Interactive Plotter
# The X-axis "Jan 01" is the point where the 'Z' contract expires
plotter.plot(
    aligned, 
    title=f"{symbol} Butterfly Seasonality: {expression}"
)

In [1]:
import pandas as pd
from database.manager import DatabaseManager
import re
import os

# 1. Setup
db = DatabaseManager()

# 2. Parameters
symbol = "CL"  # IMPORTANT: Check if your SEAC app calls it 'CL' or 'WTI' or 'CL_S'
expression = "Z26 - 2*F27 + G27"
seasons = [2023, 2024, 2025, 2026]
output_file = "MASTER_BUTTERFLY_CHECK.xlsx"

# 3. Step 1: Check what symbols actually exist in your DB
db.cursor.execute("SELECT DISTINCT symbol FROM seac_settlements LIMIT 20")
db_symbols = [row[0] for row in db.cursor.fetchall()]
print(f"Symbols found in your DB: {db_symbols}")

if symbol not in db_symbols:
    print(f"ERROR: Symbol '{symbol}' not found in Database! Use one from the list above.")
else:
    # 4. Prepare for Excel
    all_season_dfs = {}

    for s in seasons:
        print(f"\n--- Checking Season {s} ---")
        
        # Build codes: Season 2024 -> Z24, F25, G25
        matches = re.findall(r"([FGHJKMNQUVXZ])(\d{2})", expression)
        ref_year_expr = int(matches[0][1])
        season_codes = [f"{m}{(s + (int(y) - ref_year_expr)) % 100:02d}" for m, y in matches]
        
        # Fetch raw data
        raw_data_dict = db.get_contract_history(symbol, season_codes)
        
        season_df = pd.DataFrame()
        for code in season_codes:
            if code in raw_data_dict:
                count = len(raw_data_dict[code])
                print(f"  Found {count} rows for {code}")
                s_data = raw_data_dict[code][['Close']].rename(columns={'Close': code})
                if season_df.empty:
                    season_df = s_data
                else:
                    season_df = season_df.join(s_data, how='outer')
            else:
                print(f"  MISSING: No data found for {code}")

        if not season_df.empty:
            # Calculate Fly
            z, f, g = season_codes
            if z in season_df and f in season_df and g in season_df:
                season_df['BUTTERFLY_VALUE'] = season_df[z] - (2 * season_df[f]) + season_df[g]
            
            season_df.sort_index(ascending=False, inplace=True)
            all_season_dfs[f"Season_{s}"] = season_df

    # 5. Save only if we actually found data
    if all_season_dfs:
        with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
            for sheet_name, df in all_season_dfs.items():
                df.to_excel(writer, sheet_name=sheet_name)
        print(f"\nSUCCESS! File saved: {os.path.abspath(output_file)}")
    else:
        print("\nFAILED: No data was found for any of the requested seasons.")

db.close()

Symbols found in your DB: ['CL', 'CO', 'FCPO', 'G', 'GC', 'HG', 'HO', 'NG', 'NGHH', 'RB', 'SI']

--- Checking Season 2023 ---
  Found 1500 rows for Z23
  Found 500 rows for F24
  Found 500 rows for G24

--- Checking Season 2024 ---
  Found 1500 rows for Z24
  Found 500 rows for F25
  Found 500 rows for G25

--- Checking Season 2025 ---
  Found 1500 rows for Z25
  Found 500 rows for F26
  Found 500 rows for G26

--- Checking Season 2026 ---
  Found 1500 rows for Z26
  Found 500 rows for F27
  Found 500 rows for G27

SUCCESS! File saved: c:\Users\aashutosh.gandhi\OneDrive - hertshtengroup.com\Desktop\VScode\V2\MASTER_BUTTERFLY_CHECK.xlsx


In [3]:
from datetime import datetime
import pandas as pd

# 1. Setup
symbol = "CL"
expression = "Z26 - 2*F27 + G27"
seasons = [2023, 2024, 2025, 2026]

# 2. Run Pipeline
print(f"--- Data Health Report for {expression} ---")
codes = engine.get_required_codes(expression, seasons)
raw = db.get_contract_history(symbol, codes)

# Verify Raw Leg Lengths
for c in codes:
    if c in raw:
        print(f"Leg {c}: {len(raw[c])} rows")

prep = seaso.prepare_calendar(raw)
calc = seaso.calculate_expression(prep, expression)

# Verify Inner Join Result (All seasons should now have similar lengths)
print("\n--- After Expression Calculation (Inner Join) ---")
for s, df in calc.items():
    print(f"Season {s}: {len(df)} rows | Start: {df.index.min().date()} | End: {df.index.max().date()}")

# 3. Align and Plot
aligned = seaso.align_to_expiry(calc, window_days=250)

# Verify Alignment (All dates should be in year 2000)
print("\n--- After Alignment (Year 2000 Check) ---")
for s, df in aligned.items():
    print(f"Season {s} Aligned Range: {df.index.min().date()} to {df.index.max().date()}")

# 4. Final Visual Check
avg = seaso.get_average(aligned)
if not avg.empty:
    aligned["AVERAGE"] = avg

plotter.plot(aligned, title=f"Verified SEAC Seasonality: {expression}")

--- Data Health Report for Z26 - 2*F27 + G27 ---
Leg G26: 500 rows
Leg G24: 500 rows
Leg Z25: 1500 rows
Leg F27: 500 rows
Leg F24: 500 rows
Leg G27: 500 rows
Leg F25: 500 rows
Leg F26: 500 rows
Leg Z26: 1500 rows
Leg Z23: 1500 rows
Leg Z24: 1500 rows
Leg G25: 500 rows

--- After Expression Calculation (Inner Join) ---
Season 2023: 665 rows | Start: 2022-01-25 | End: 2023-11-20
Season 2024: 666 rows | Start: 2023-01-25 | End: 2024-11-20
Season 2025: 667 rows | Start: 2024-01-24 | End: 2025-11-20
Season 2026: 725 rows | Start: 2024-07-30 | End: 2026-07-24

--- After Alignment (Year 2000 Check) ---
Season 2023 Aligned Range: 1999-04-26 to 2000-02-01
Season 2024 Aligned Range: 1999-04-26 to 2000-02-01
Season 2025 Aligned Range: 1999-04-26 to 2000-02-01
Season 2026 Aligned Range: 1999-04-26 to 2000-02-01


In [3]:
# --- Cell 1: setup ---
from database.manager import DatabaseManager
from analysis.engine import Engine
from analysis.seasonality import Seasonality
from analysis.plotting import SeasonalityPlotter

SYMBOL     = "CL"
EXPRESSION = "Z26 - 2*F27 + G27"
SEASONS    = list(range(2015, 2027))   # years to overlay
WINDOW_DAYS = 365

In [4]:
# --- Cell 2: fetch only the contracts this expression actually needs ---
engine = Engine()
codes = engine.get_required_codes(EXPRESSION, SEASONS)

db = DatabaseManager()
raw_data = db.get_contract_history(SYMBOL, codes)
db.close()

missing = [c for c in codes if c not in raw_data]
if missing:
    print(f"⚠️ no data for: {missing}")
print({k: len(v) for k, v in raw_data.items()})

{'G21': 500, 'G19': 500, 'G26': 500, 'F16': 500, 'G25': 500, 'Z20': 1500, 'Z19': 1500, 'Z16': 1500, 'Z18': 1500, 'G23': 500, 'F26': 500, 'Z23': 1500, 'Z25': 1500, 'F19': 500, 'F25': 500, 'Z22': 1500, 'F17': 500, 'G16': 500, 'F20': 500, 'Z24': 1500, 'F27': 500, 'Z17': 1500, 'F18': 500, 'G22': 500, 'F23': 500, 'Z15': 1500, 'G27': 500, 'F21': 500, 'G20': 500, 'G24': 500, 'G18': 500, 'F22': 500, 'G17': 500, 'Z21': 1500, 'Z26': 1500, 'F24': 500}


In [5]:
# --- Cell 3: RAW verification plot — real closes, no interpolation, no join ---
plotter = SeasonalityPlotter()
plotter.plot(raw_data, title=f"Raw closes — legs for '{EXPRESSION}'")

In [8]:
# --- Cell 3b: each outright leg plotted separately, raw closes, no interpolation ---
import plotly.graph_objects as go

for code, df in raw_data.items():
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=df.index, y=df['Close'],
        mode='lines+markers', name=code,
        line=dict(width=2, color='#00CC96'),
        marker=dict(size=4),
    ))
    fig.update_layout(
        template="plotly_dark", paper_bgcolor='black', plot_bgcolor='black',
        title=f"{SYMBOL} {code} — raw settlement closes ({len(df)} rows)",
        xaxis=dict(tickformat='%Y-%m-%d', gridcolor='#333'),
        yaxis=dict(title="Price", gridcolor='#333'),
        hovermode="x unified",
    )
    fig.show()

In [6]:
# --- Cell 4: compute the spread per season (feed RAW data, not prepare_calendar output) ---
seas = Seasonality()
season_results = seas.calculate_expression(raw_data, EXPRESSION)
print("seasons with valid overlap:", sorted(season_results.keys()))

seasons with valid overlap: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]


In [7]:
# --- Cell 5: align onto common calendar-month axis + average, then plot ---
aligned = seas.align_to_expiry(season_results, window_days=WINDOW_DAYS)
aligned["AVERAGE"] = seas.get_average(aligned)

plotter.plot(aligned, title=f"Seasonality: {EXPRESSION}")

In [9]:
# --- Cell 3c-check: empirical expiry estimates per leg ---
from analysis.expiry import ExpiryEstimator

db = DatabaseManager()
estimator = ExpiryEstimator(db)

expiry_map = {}
for code in raw_data.keys():
    est, offsets = estimator.estimate_expiry(SYMBOL, code)
    expiry_map[code] = est
    print(f"{code}: estimated expiry={est.date() if est is not None else 'N/A'}, "
          f"based on {len(offsets)} historical contracts, "
          f"offset samples (days before delivery-month-1st)={offsets}")

db.close()

G21: estimated expiry=2021-01-20, based on 15 historical contracts, offset samples (days before delivery-month-1st)=[12, 12, 10, 11, 12, 12, 12, 10, 10, 11, 12, 12, 10, 11, 12]
G19: estimated expiry=2019-01-20, based on 15 historical contracts, offset samples (days before delivery-month-1st)=[12, 12, 10, 11, 12, 12, 12, 10, 11, 12, 12, 12, 10, 11, 12]
G26: estimated expiry=2026-01-20, based on 15 historical contracts, offset samples (days before delivery-month-1st)=[12, 12, 10, 11, 12, 12, 12, 10, 10, 11, 12, 12, 12, 10, 11]
F16: estimated expiry=2015-12-19, based on 15 historical contracts, offset samples (days before delivery-month-1st)=[12, 12, 13, 13, 13, 12, 13, 13, 13, 11, 12, 12, 13, 13, 13]
G25: estimated expiry=2025-01-20, based on 15 historical contracts, offset samples (days before delivery-month-1st)=[12, 12, 10, 11, 12, 12, 12, 10, 10, 11, 12, 12, 12, 10, 12]
Z20: estimated expiry=2020-11-19, based on 14 historical contracts, offset samples (days before delivery-month-1st)

In [11]:
# --- Cell 3d: each outright, windowed to 400 calendar days before its estimated expiry ---
from analysis.expiry import ExpiryEstimator
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go

WINDOW_DAYS = 450
today = pd.Timestamp(datetime.now().date())

db = DatabaseManager()
estimator = ExpiryEstimator(db)

for code, df in raw_data.items():
    expiry, offsets = estimator.estimate_expiry(SYMBOL, code)

    if expiry is None:
        print(f"⚠️ {code}: not enough historical same-month contracts to estimate expiry — skipping")
        continue

    window_start = expiry - pd.Timedelta(days=WINDOW_DAYS)
    window_end = min(expiry, today)  # unexpired contracts can't show data past today
    status = "expired" if expiry < today else "live"

    windowed = df[(df.index >= window_start) & (df.index <= window_end)]

    print(f"{code}: expiry~{expiry.date()} ({status}), "
          f"window=[{window_start.date()} → {window_end.date()}], "
          f"rows in window={len(windowed)} (of {len(df)} total), "
          f"n_offset_samples={len(offsets)}")

    if windowed.empty:
        print(f"  ⚠️ no data in window for {code}")
        continue

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=windowed.index, y=windowed['Close'],
        mode='lines+markers', name=code,
        line=dict(width=2, color='#00CC96'),
        marker=dict(size=4),
    ))
    fig.update_layout(
        template="plotly_dark", paper_bgcolor='black', plot_bgcolor='black',
        title=f"{SYMBOL} {code} — {status}, expiry~{expiry.date()}, {len(windowed)} rows",
        xaxis=dict(tickformat='%Y-%m-%d', gridcolor='#333'),
        yaxis=dict(title="Price", gridcolor='#333'),
        hovermode="x unified",
    )
    fig.show()

db.close()

G21: expiry~2021-01-20 (expired), window=[2019-10-28 → 2021-01-20], rows in window=309 (of 500 total), n_offset_samples=15


G19: expiry~2019-01-20 (expired), window=[2017-10-27 → 2019-01-20], rows in window=308 (of 500 total), n_offset_samples=15


G26: expiry~2026-01-20 (expired), window=[2024-10-27 → 2026-01-20], rows in window=308 (of 500 total), n_offset_samples=15


F16: expiry~2015-12-19 (expired), window=[2014-09-25 → 2015-12-19], rows in window=311 (of 500 total), n_offset_samples=15


G25: expiry~2025-01-20 (expired), window=[2023-10-28 → 2025-01-20], rows in window=307 (of 500 total), n_offset_samples=15


...

F22: expiry~2021-12-19 (expired), window=[2020-09-25 → 2021-12-19], rows in window=310 (of 500 total), n_offset_samples=15


G17: expiry~2017-01-20 (expired), window=[2015-10-28 → 2017-01-20], rows in window=309 (of 500 total), n_offset_samples=15


Z21: expiry~2021-11-19 (expired), window=[2020-08-26 → 2021-11-19], rows in window=312 (of 1500 total), n_offset_samples=14


Z26: expiry~2026-11-19 (live), window=[2025-08-26 → 2026-07-27], rows in window=229 (of 1500 total), n_offset_samples=15


F24: expiry~2023-12-19 (expired), window=[2022-09-25 → 2023-12-19], rows in window=311 (of 500 total), n_offset_samples=15


In [12]:
# --- Cell 3e: one plot per month letter, all historical years overlaid, aligned to expiry ---
import re
import pandas as pd
import plotly.graph_objects as go
from analysis.expiry import ExpiryEstimator

WINDOW_DAYS = 400

db = DatabaseManager()
estimator = ExpiryEstimator(db)

# Discover all contract codes we have for this symbol
db.cursor.execute("SELECT DISTINCT contract_code FROM seac_settlements WHERE symbol = ?", (SYMBOL,))
all_codes = [r[0] for r in db.cursor.fetchall()]

# Group by month letter
by_letter = {}
for code in all_codes:
    letter = code[0]
    by_letter.setdefault(letter, []).append(code)

today = pd.Timestamp(pd.Timestamp.now().date())
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A', '#19D3F3',
          '#FF6692', '#B6E880', '#FF97FF', '#FECB52']

for letter, codes in sorted(by_letter.items()):
    hist = db.get_contract_history(SYMBOL, codes)  # {code: df}

    aligned_series = {}
    for code, df in sorted(hist.items()):
        expiry, offsets = estimator.estimate_expiry(SYMBOL, code)
        if expiry is None or df.empty:
            continue

        window_start = expiry - pd.Timedelta(days=WINDOW_DAYS)
        window_end = min(expiry, today)
        windowed = df[(df.index >= window_start) & (df.index <= window_end)]
        if windowed.empty:
            continue

        # Align x-axis to "days to expiry" (negative = before expiry, 0 = expiry)
        days_to_expiry = (windowed.index - expiry).days
        year = 2000 + int(code[1:])
        aligned_series[year] = pd.Series(windowed['Close'].values, index=days_to_expiry)

    if not aligned_series:
        print(f"⚠️ {letter}: no valid windows found — skipping")
        continue

    fig = go.Figure()
    for i, (year, s) in enumerate(sorted(aligned_series.items())):
        fig.add_trace(go.Scatter(
            x=s.index, y=s.values, mode='lines', name=str(year),
            line=dict(width=1.5, color=colors[i % len(colors)]),
            hovertemplate=f"{year}<br>Days to expiry: %{{x}}<br>Price: %{{y:.2f}}<extra></extra>"
        ))

    # Average across years, aligned on common day-offset grid
    combined = pd.concat(
        [s.rename(y) for y, s in aligned_series.items()], axis=1
    ).sort_index()
    avg = combined.mean(axis=1)
    fig.add_trace(go.Scatter(
        x=avg.index, y=avg.values, mode='lines', name='AVERAGE',
        line=dict(width=4, color='white'),
        hovertemplate=f"AVERAGE<br>Days to expiry: %{{x}}<br>Price: %{{y:.2f}}<extra></extra>"
    ))

    fig.update_layout(
        template="plotly_dark", paper_bgcolor='black', plot_bgcolor='black',
        title=f"{SYMBOL} {letter} contracts — {len(aligned_series)} years, {WINDOW_DAYS}d pre-expiry",
        xaxis=dict(title="Days to expiry", gridcolor='#333'),
        yaxis=dict(title="Price", gridcolor='#333'),
        hovermode="x unified",
    )
    fig.show()

db.close()

In [13]:
# --- Cell 3f: CL Z-contract seasonality, last 10 years, aligned to expiry ---
import pandas as pd
import plotly.graph_objects as go
from analysis.expiry import ExpiryEstimator

LETTER = "Z"
N_YEARS = 10
WINDOW_DAYS = 400

db = DatabaseManager()
estimator = ExpiryEstimator(db)

db.cursor.execute(
    "SELECT DISTINCT contract_code FROM seac_settlements WHERE symbol = ? AND contract_code LIKE ?",
    (SYMBOL, f"{LETTER}%")
)
codes = [r[0] for r in db.cursor.fetchall()]

# Keep only the most recent N_YEARS by contract year
codes_with_year = sorted(codes, key=lambda c: int(c[1:]))
codes = codes_with_year[-N_YEARS:]

hist = db.get_contract_history(SYMBOL, codes)

today = pd.Timestamp(pd.Timestamp.now().date())
colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A',
          '#19D3F3', '#FF6692', '#B6E880', '#FF97FF', '#FECB52']

aligned_series = {}
for code, df in sorted(hist.items()):
    expiry, offsets = estimator.estimate_expiry(SYMBOL, code)
    if expiry is None or df.empty:
        print(f"⚠️ {code}: skipped (no expiry estimate or no data)")
        continue

    window_start = expiry - pd.Timedelta(days=WINDOW_DAYS)
    window_end = min(expiry, today)
    windowed = df[(df.index >= window_start) & (df.index <= window_end)]
    if windowed.empty:
        print(f"⚠️ {code}: no data in window")
        continue

    days_to_expiry = (windowed.index - expiry).days
    year = 2000 + int(code[1:])
    aligned_series[year] = pd.Series(windowed['Close'].values, index=days_to_expiry)
    print(f"{code}: expiry~{expiry.date()}, rows={len(windowed)}")

db.close()

fig = go.Figure()
for i, (year, s) in enumerate(sorted(aligned_series.items())):
    fig.add_trace(go.Scatter(
        x=s.index, y=s.values, mode='lines', name=str(year),
        line=dict(width=1.5, color=colors[i % len(colors)]),
        hovertemplate=f"{year}<br>Days to expiry: %{{x}}<br>Price: %{{y:.2f}}<extra></extra>"
    ))

combined = pd.concat([s.rename(y) for y, s in aligned_series.items()], axis=1).sort_index()
avg = combined.mean(axis=1)
fig.add_trace(go.Scatter(
    x=avg.index, y=avg.values, mode='lines', name='AVERAGE',
    line=dict(width=4, color='white'),
    hovertemplate=f"AVERAGE<br>Days to expiry: %{{x}}<br>Price: %{{y:.2f}}<extra></extra>"
))

fig.update_layout(
    template="plotly_dark", paper_bgcolor='black', plot_bgcolor='black',
    title=f"{SYMBOL} {LETTER} — last {len(aligned_series)} years, {WINDOW_DAYS}d pre-expiry seasonality",
    xaxis=dict(title="Days to expiry", gridcolor='#333'),
    yaxis=dict(title="Price", gridcolor='#333'),
    hovermode="x unified",
)
fig.show()

Z21: expiry~2021-11-19, rows=277
Z22: expiry~2022-11-19, rows=276
Z23: expiry~2023-11-19, rows=275
Z24: expiry~2024-11-19, rows=276
Z25: expiry~2025-11-19, rows=276
Z26: expiry~2026-11-19, rows=194
⚠️ Z27: no data in window
⚠️ Z28: no data in window
⚠️ Z29: no data in window
⚠️ Z30: no data in window


In [15]:
# --- Cell 3g: CL Z seasonality, aligned by trading-day index (fixes the zigzag), month x-axis ---
import pandas as pd
import plotly.graph_objects as go
from analysis.expiry import ExpiryEstimator

LETTER = "Z"
N_YEARS = 10
WINDOW_DAYS = 400

db = DatabaseManager()
estimator = ExpiryEstimator(db)

db.cursor.execute(
    "SELECT DISTINCT contract_code FROM seac_settlements WHERE symbol = ? AND contract_code LIKE ?",
    (SYMBOL, f"{LETTER}%")
)
codes = sorted([r[0] for r in db.cursor.fetchall()], key=lambda c: int(c[1:]))[-N_YEARS:]
hist = db.get_contract_history(SYMBOL, codes)
today = pd.Timestamp(pd.Timestamp.now().date())

aligned_series = {}
ref_ticks = None  # (trading_day_index list, real dates) from longest EXPIRED year, for month labels

for code, df in sorted(hist.items()):
    expiry, _ = estimator.estimate_expiry(SYMBOL, code)
    if expiry is None or df.empty:
        continue
    window_start = expiry - pd.Timedelta(days=WINDOW_DAYS)
    window_end = min(expiry, today)
    windowed = df[(df.index >= window_start) & (df.index <= window_end)].sort_index()
    if windowed.empty:
        continue

    n = len(windowed)
    trading_day_index = list(range(-(n - 1), 1))  # last row -> 0, one before -> -1, etc.
    year = 2000 + int(code[1:])
    aligned_series[year] = pd.Series(windowed['Close'].values, index=trading_day_index)

    if expiry < today and (ref_ticks is None or n > len(ref_ticks[0])):
        ref_ticks = (trading_day_index, windowed.index)

db.close()

colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A',
          '#19D3F3', '#FF6692', '#B6E880', '#FF97FF', '#FECB52']

fig = go.Figure()
for i, (year, s) in enumerate(sorted(aligned_series.items())):
    fig.add_trace(go.Scatter(
        x=s.index, y=s.values, mode='lines', name=str(year),
        line=dict(width=1.5, color=colors[i % len(colors)]),
        hovertemplate=f"{year}<br>Trading day: %{{x}}<br>Price: %{{y:.2f}}<extra></extra>"
    ))

combined = pd.concat([s.rename(y) for y, s in aligned_series.items()], axis=1).sort_index()
avg = combined.mean(axis=1)
fig.add_trace(go.Scatter(
    x=avg.index, y=avg.values, mode='lines', name='AVERAGE',
    line=dict(width=4, color='white'),
    hovertemplate=f"AVERAGE<br>Trading day: %{{x}}<br>Price: %{{y:.2f}}<extra></extra>"
))

# Month tick labels using one real expired year's calendar as reference
tickvals, ticktext = [], []
if ref_ticks:
    idxs, dates = ref_ticks
    seen = set()
    for idx, date in zip(idxs, dates):
        key = (date.year, date.month)
        if key not in seen:
            seen.add(key)
            tickvals.append(idx)
            ticktext.append(date.strftime('%b'))

fig.update_layout(
    template="plotly_dark", paper_bgcolor='black', plot_bgcolor='black',
    title=f"{SYMBOL} {LETTER} — last {len(aligned_series)} years, {WINDOW_DAYS}d pre-expiry (trading-day aligned)",
    xaxis=dict(title="Month before expiry", tickvals=tickvals, ticktext=ticktext, gridcolor='#333'),
    yaxis=dict(title="Price", gridcolor='#333'),
    hovermode="x unified",
)
fig.show()

In [16]:
# --- Cell 3h: CL Z seasonality, trading-day aligned to actual expiry (fixes both zigzag and live-year shift) ---
import pandas as pd
import plotly.graph_objects as go
from analysis.expiry import ExpiryEstimator

LETTER = "Z"
N_YEARS = 6
WINDOW_DAYS = 400

db = DatabaseManager()
estimator = ExpiryEstimator(db)

db.cursor.execute(
    "SELECT DISTINCT contract_code FROM seac_settlements WHERE symbol = ? AND contract_code LIKE ?",
    (SYMBOL, f"{LETTER}%")
)
codes = sorted([r[0] for r in db.cursor.fetchall()], key=lambda c: int(c[1:]))[-N_YEARS:]
hist = db.get_contract_history(SYMBOL, codes)
today = pd.Timestamp(pd.Timestamp.now().date())

aligned_series = {}
ref_ticks = None  # for month x-axis labels, taken from longest EXPIRED year

for code, df in sorted(hist.items()):
    expiry, _ = estimator.estimate_expiry(SYMBOL, code)
    if expiry is None or df.empty:
        continue
    window_start = expiry - pd.Timedelta(days=WINDOW_DAYS)
    window_end = min(expiry, today)
    windowed = df[(df.index >= window_start) & (df.index <= window_end)].sort_index()
    if windowed.empty:
        continue

    # Canonical business-day calendar ending AT expiry (0 = expiry), weekends excluded
    full_bdays = pd.bdate_range(start=window_start, end=expiry)
    pos_map = {d: i - (len(full_bdays) - 1) for i, d in enumerate(full_bdays)}

    idx_vals, price_vals = [], []
    for d, price in zip(windowed.index, windowed['Close'].values):
        if d in pos_map:
            idx_vals.append(pos_map[d])
            price_vals.append(price)

    if not idx_vals:
        continue

    year = 2000 + int(code[1:])
    aligned_series[year] = pd.Series(price_vals, index=idx_vals)

    if expiry < today and (ref_ticks is None or len(idx_vals) > len(ref_ticks[0])):
        ref_ticks = (idx_vals, windowed.index)

db.close()

colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A',
          '#19D3F3', '#FF6692', '#B6E880', '#FF97FF', '#FECB52']

fig = go.Figure()
for i, (year, s) in enumerate(sorted(aligned_series.items())):
    fig.add_trace(go.Scatter(
        x=s.index, y=s.values, mode='lines', name=str(year),
        line=dict(width=1.5, color=colors[i % len(colors)]),
        hovertemplate=f"{year}<br>Trading day: %{{x}}<br>Price: %{{y:.2f}}<extra></extra>"
    ))

combined = pd.concat([s.rename(y) for y, s in aligned_series.items()], axis=1).sort_index()
avg = combined.mean(axis=1)
fig.add_trace(go.Scatter(
    x=avg.index, y=avg.values, mode='lines', name='AVERAGE',
    line=dict(width=4, color='white'),
    hovertemplate=f"AVERAGE<br>Trading day: %{{x}}<br>Price: %{{y:.2f}}<extra></extra>"
))

tickvals, ticktext = [], []
if ref_ticks:
    idxs, dates = ref_ticks
    seen = set()
    for idx, date in zip(idxs, dates):
        key = (date.year, date.month)
        if key not in seen:
            seen.add(key)
            tickvals.append(idx)
            ticktext.append(date.strftime('%b'))

fig.update_layout(
    template="plotly_dark", paper_bgcolor='black', plot_bgcolor='black',
    title=f"{SYMBOL} {LETTER} — last {len(aligned_series)} years, {WINDOW_DAYS}d pre-expiry (expiry-anchored)",
    xaxis=dict(title="Month before expiry", tickvals=tickvals, ticktext=ticktext, gridcolor='#333'),
    yaxis=dict(title="Price", gridcolor='#333'),
    hovermode="x unified",
)
fig.show()

In [19]:
# --- Cell 3j: CL Z seasonality, year range + weekend interpolation (fixed alignment) ---
import pandas as pd
import plotly.graph_objects as go
from analysis.expiry import ExpiryEstimator

LETTER = "Z"
START_YEAR = 2019
END_YEAR = 2026
WINDOW_DAYS = 400

db = DatabaseManager()
estimator = ExpiryEstimator(db)

db.cursor.execute(
    "SELECT DISTINCT contract_code FROM seac_settlements WHERE symbol = ? AND contract_code LIKE ?",
    (SYMBOL, f"{LETTER}%")
)
all_codes = [r[0] for r in db.cursor.fetchall()]
codes = sorted(
    [c for c in all_codes if START_YEAR <= 2000 + int(c[1:]) <= END_YEAR],
    key=lambda c: int(c[1:])
)
if not codes:
    print(f"⚠️ no contracts found for {LETTER} in [{START_YEAR}, {END_YEAR}]")

hist = db.get_contract_history(SYMBOL, codes)
today = pd.Timestamp(pd.Timestamp.now().date())

aligned_series = {}
ref_dates_for_ticks = None

for code, df in sorted(hist.items()):
    expiry, _ = estimator.estimate_expiry(SYMBOL, code)
    if expiry is None or df.empty:
        print(f"⚠️ {code}: skipped (no expiry estimate or no data)")
        continue

    # Normalize everything to plain dates (strips any hidden time/dtype mismatch)
    df = df.copy()
    df.index = pd.to_datetime(df.index).normalize()
    expiry = pd.Timestamp(expiry).normalize()
    window_start = (expiry - pd.Timedelta(days=WINDOW_DAYS)).normalize()
    window_end = min(expiry, today).normalize()

    raw_windowed = df[(df.index >= window_start) & (df.index <= window_end)].sort_index()
    raw_windowed = raw_windowed[~raw_windowed.index.duplicated(keep='last')]

    if raw_windowed.empty:
        print(f"⚠️ {code}: no data in window")
        continue

    full_calendar = pd.date_range(window_start, window_end, freq='D')
    overlap = raw_windowed.index.intersection(full_calendar)

    filled = raw_windowed.reindex(full_calendar).interpolate(method='linear').dropna()

    print(f"{code}: expiry~{expiry.date()}, raw rows={len(raw_windowed)}, "
          f"overlap={len(overlap)}, interpolated to {len(filled)} continuous days")

    if filled.empty:
        print(f"  ⚠️ {code}: still empty after interpolation — check overlap count above")
        continue

    days_to_expiry = (filled.index - expiry).days
    year = 2000 + int(code[1:])
    aligned_series[year] = pd.Series(filled['Close'].values, index=days_to_expiry)

    if expiry < today and (ref_dates_for_ticks is None or len(filled) > len(ref_dates_for_ticks[0])):
        ref_dates_for_ticks = (days_to_expiry, filled.index)

db.close()

if not aligned_series:
    print("⚠️ nothing to plot — all contracts failed alignment, see warnings above")
else:
    colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA', '#FFA15A',
              '#19D3F3', '#FF6692', '#B6E880', '#FF97FF', '#FECB52']

    fig = go.Figure()
    for i, (year, s) in enumerate(sorted(aligned_series.items())):
        fig.add_trace(go.Scatter(
            x=s.index, y=s.values, mode='lines', name=str(year),
            line=dict(width=1.5, color=colors[i % len(colors)]),
            hovertemplate=f"{year}<br>Days to expiry: %{{x}}<br>Price: %{{y:.2f}}<extra></extra>"
        ))

    combined = pd.concat([s.rename(y) for y, s in aligned_series.items()], axis=1).sort_index()
    avg = combined.mean(axis=1)
    fig.add_trace(go.Scatter(
        x=avg.index, y=avg.values, mode='lines', name='AVERAGE',
        line=dict(width=4, color='white'),
        hovertemplate=f"AVERAGE<br>Days to expiry: %{{x}}<br>Price: %{{y:.2f}}<extra></extra>"
    ))

    tickvals, ticktext = [], []
    if ref_dates_for_ticks:
        offsets, dates = ref_dates_for_ticks
        seen = set()
        for off, date in zip(offsets, dates):
            key = (date.year, date.month)
            if key not in seen:
                seen.add(key)
                tickvals.append(off)
                ticktext.append(date.strftime('%b'))

    fig.update_layout(
        template="plotly_dark", paper_bgcolor='black', plot_bgcolor='black',
        title=f"{SYMBOL} {LETTER} — {START_YEAR}-{END_YEAR}, {WINDOW_DAYS}d pre-expiry (weekends interpolated)",
        xaxis=dict(title="Month before expiry", tickvals=tickvals, ticktext=ticktext, gridcolor='#333'),
        yaxis=dict(title="Price", gridcolor='#333'),
        hovermode="x unified",
    )
    fig.show()

Z19: expiry~2019-11-19, raw rows=278, overlap=278, interpolated to 401 continuous days
Z20: expiry~2020-11-19, raw rows=278, overlap=278, interpolated to 401 continuous days
Z21: expiry~2021-11-19, raw rows=278, overlap=278, interpolated to 401 continuous days
Z22: expiry~2022-11-19, raw rows=277, overlap=277, interpolated to 401 continuous days
Z23: expiry~2023-11-19, raw rows=275, overlap=275, interpolated to 399 continuous days
Z24: expiry~2024-11-19, raw rows=277, overlap=277, interpolated to 401 continuous days
Z25: expiry~2025-11-19, raw rows=277, overlap=277, interpolated to 401 continuous days
Z26: expiry~2026-11-19, raw rows=195, overlap=195, interpolated to 286 continuous days


In [1]:
# --- setup (once) ---
from database.manager import DatabaseManager
from analysis.seasonality import Seasonality
from analysis.plotting import SeasonalityPlotter

db = DatabaseManager()
sznlty = Seasonality(db)
plotter = SeasonalityPlotter()

In [2]:
result = sznlty.outright_seasonality("CO", "Z", start_year=2019, end_year=2026, window_days=400)
plotter.plot_seasonality(result, title="CO Z — outright seasonality")

In [4]:
result = sznlty.expression_seasonality("CL", "X26-Z26", start_year=2012, end_year=2026, window_days=400)
plotter.plot_seasonality(result, title="CL X-Z spread — seasonality")